# Reproducing the computational results

Run top to bottom. Paths resolve relative to this notebook, so nothing needs
editing. Every solve uses `glpk_exact`; see the deposit README for why the
default GLPK interface must not be used here.

The notebook runs in nine steps:

| | | |
|---|---|---|
| 1 | CHO-K1, scenario 52 | coupling |
| 2 | CHO-K1, BCAT1 knockout with mild hypothermia in place of the growth regulations | coupling |
| 3 | CHO-K1 with BCAT2 added, scenario 52 | coupling lost |
| 4 | iCHO3K, scenario 52 | coupling lost |
| 5 | iCHO3K, scenario 52 with BCAT2 and the leucine sinks knocked out | coupling |
| 6 | iCHO3K, which leucine routes are genuine sinks | one drain, twelve futile cycles |
| 7 | Leucine thresholds for coupling, CHO-K1 | threshold = forced uptake / 0.540177 |
| 8 | Sensitivity to the upregulation coefficient | infeasible from e_up 0.61 |
| 9 | Growth coupling across every scenario | 89 of 89 forced |

Throughout, coupling means the **minimum** feasible IgG flux at a fixed growth
rate is above zero, so production is forced rather than merely permitted. A
single FBA solution would not show this.


## Setup

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import scipy.io as sio
from cobra import Reaction, Metabolite
from cobra.io import load_matlab_model
import subprocess, sys

# Exact arithmetic throughout. Plain GLPK returns solutions that violate the IgG
# demand upper bound once the design regulations are applied - see the README.
SOLVER = "glpk_exact"
QCONV  = 0.315 * 24 * 150000          # flux -> pg/cell/day
IGG, BIO = "DM_igg[g]", "biomass_cho_producing"
DEP = Path.cwd().parent
SETS = {"D1": "FVA_EX_SUE_K1_NoEPO.mat",         "D2": "FVA_EX_SAR_FeedC_K1_NoEPO.mat",
        "D3": "FVA_EX_SAR_FeedAll_K1_NoEPO.mat", "D4": "FVA_EX_SAR_FeedAll40_K1_NoEPO.mat"}

K1 = load_matlab_model(str(DEP/"models"/"iCHO_K1_used.mat")); K1.solver = SOLVER
I3 = load_matlab_model(str(DEP/"models"/"iCHO3K.mat"));       I3.solver = SOLVER
REF = [r.id for r in K1.reactions]
BND = {d: sio.loadmat(str(DEP/"constraints"/f), squeeze_me=True)["FVA_EX"]
       for d, f in SETS.items()}
UPTAKE = {d: abs(float(BND[d][REF.index("EX_leu_L_e_"), 1])) for d in SETS}

def k1_setup(m, ds, block_cho=True):
    """Dataset bounds on every reaction; objectives reopened."""
    for i, r in enumerate(m.reactions):
        r.bounds = (float(BND[ds][i, 0]), float(BND[ds][i, 1]))
    m.reactions.get_by_id(IGG).bounds = (0., 100.)
    m.reactions.get_by_id(BIO).bounds = (0., 100.)
    if block_cho:
        m.reactions.get_by_id("biomass_cho").bounds = (0., 0.)
    m.objective = m.reactions.get_by_id(BIO)

def scenario52(m, ds):
    """The seven regulations, RKO_Results.m line 712, e_up = 0.60.

    RTOTAL2t_b and RTOTAL3t_b are the export directions. In the irreversible
    model the upregulation raises the lower bound of the _b split to 60% of its
    range; on this reversible model that is an upper bound of ub + 0.60*lb.
    """
    for rid in ("THRA", "AACOAT", "LEUTA", "PSP_L"):
        m.reactions.get_by_id(rid).bounds = (0., 0.)
    r = m.reactions.get_by_id("r1109")
    if r.lower_bound < 0:
        r.lower_bound = 0.                       # r1109_b knockout
    else:
        print("   note: r1109 is irreversible here, so the r1109_b knockout is inert")
    for rid in ("RTOTAL2t", "RTOTAL3t"):
        i = REF.index(rid)
        lb, ub = float(BND[ds][i, 0]), float(BND[ds][i, 1])
        m.reactions.get_by_id(rid).bounds = (lb, ub + 0.60 * lb)

def forced(m, bio_id, igg_id, mu, label):
    """Minimum IgG at a fixed growth rate, and the width of the feasible interval.

    Only a minimum above zero demonstrates coupling. Minimising and maximising the
    demand reaction directly gives both bounds in two linear programmes.
    """
    m.reactions.get_by_id(bio_id).bounds = (mu * (1 - 1e-9), mu)
    m.objective = m.reactions.get_by_id(igg_id)
    m.objective.direction = "min"
    lo = m.slim_optimize()
    if lo != lo:
        raise ValueError("infeasible at this growth rate")
    m.objective.direction = "max"
    hi = m.slim_optimize()
    hi = lo if hi != hi else hi
    print(f"   {label:38s} mu {mu:.9f}   qmAb {lo*QCONV:9.4f}   interval width {hi-lo:.1e}")
    return lo


def min_igg(m, bio_id, igg_id, mu):
    """Minimum IgG at a fixed growth rate: one linear programme, no interval."""
    m.reactions.get_by_id(bio_id).bounds = (mu * (1 - 1e-9), mu)
    m.objective = m.reactions.get_by_id(igg_id)
    m.objective.direction = "min"
    v = m.slim_optimize()
    if v != v:
        raise ValueError("infeasible at this growth rate")
    return v
def add_bcat2(m, capacity):
    """BCAT2 is absent from this reconstruction. Add mitochondrial leucine and
    the two reactions that restore the route, capped at `capacity`."""
    leu_m = Metabolite("leu_L[m]", compartment="m")
    t = Reaction("LEUt5m"); t.bounds = (-1000., 1000.)
    t.add_metabolites({m.metabolites.get_by_id("leu_L[c]"): -1, leu_m: 1})
    a = Reaction("LEUTAm"); a.bounds = (-capacity, capacity)
    a.add_metabolites({m.metabolites.get_by_id("akg[m]"): -1, leu_m: -1,
                       m.metabolites.get_by_id("4mop[m]"): 1,
                       m.metabolites.get_by_id("glu_L[m]"): 1})
    m.add_reactions([t, a])

print("CHO-K1 :", len(K1.reactions), "reactions,", len(K1.genes), "genes")
print("iCHO3K :", len(I3.reactions), "reactions,", len(I3.genes), "genes")
print("forced minimum leucine uptake:", {d: round(v, 6) for d, v in UPTAKE.items()})

No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.


Using regular expression found the following compartments:c, e, g, im, l, m, n, r, x


CHO-K1 : 3952 reactions, 1202 genes
iCHO3K : 8368 reactions, 2929 genes
forced minimum leucine uptake: {'D1': 0.012905, 'D2': 0.008593, 'D3': 0.00603, 'D4': 0.010148}


## Step 1 - CHO-K1, scenario 52 and the Bcat1 gene-protein-reaction set

The seven regulations of scenario 52 under dataset D4, reproducing the predicted
operating point. The FVA interval width shows the solution is unique, so the
numbers do not depend on which vertex the solver returns.

In iCHO1766-K1 the gene Bcat1 carries three reactions, so a gene disruption
removes more than the single reaction the design specifies. The second block
evaluates the design as implemented, with all three closed.


In [2]:
with K1 as m:
    k1_setup(m, "D4", block_cho=True)
    scenario52(m, "D4")
    forced(m, BIO, IGG, m.slim_optimize(), "scenario 52")
print("   published:                             mu 0.012369243   qmAb   43.6783")

# Bcat1 governs three reactions, and a gene disruption removes all of them.
print("\ngene 100763658 governs:",
      ", ".join(sorted(r.id for r in K1.genes.get_by_id("100763658").reactions)))


def scenario52_bcat(m, ds, bcat=("LEUTA",)):
    """Scenario 52 with the Bcat1 reaction set as a parameter."""
    for rid in ("THRA", "AACOAT", "PSP_L") + tuple(bcat):
        m.reactions.get_by_id(rid).bounds = (0., 0.)
    r = m.reactions.get_by_id("r1109")
    if r.lower_bound < 0:
        r.lower_bound = 0.
    for rid in ("RTOTAL2t", "RTOTAL3t"):
        i = REF.index(rid)
        lb, ub = float(BND[ds][i, 0]), float(BND[ds][i, 1])
        m.reactions.get_by_id(rid).bounds = (lb, ub + 0.60 * lb)


for ds in ("D4", "D2"):
    for label, bcat in (("LEUTA only", ("LEUTA",)),
                        ("all three", ("LEUTA", "ILETA", "VALTA"))):
        with K1 as m:
            k1_setup(m, ds, block_cho=False)
            scenario52_bcat(m, ds, bcat)
            forced(m, BIO, IGG, m.slim_optimize(), f"{ds} {label}")


   scenario 52                            mu 0.012369243   qmAb   43.6783   interval width 9.8e-14
   published:                             mu 0.012369243   qmAb   43.6783

gene 100763658 governs: ILETA, LEUTA, VALTA


   D4 LEUTA only                          mu 0.012369243   qmAb   43.6783   interval width 9.8e-14


   D4 all three                           mu 0.006191694   qmAb   85.7241   interval width 5.1e-14


   D2 LEUTA only                          mu 0.018408720   qmAb    0.0000   interval width 1.4e-13


   D2 all three                           mu 0.008734882   qmAb   48.8162   interval width 6.7e-14


## Step 2 - CHO-K1, BCAT1 knockout with mild hypothermia

Scenario 52's six growth-suppressing regulations are replaced by a direct cap on
growth, which is what mild hypothermia does experimentally. The BCAT1 knockout is
kept. Wild type at the same cap produces no antibody; the knockout is forced to.


In [3]:
CAP = 0.0125
for label, ko in (("wild type", ()), ("BCAT1 knockout", ("LEUTA",))):
    with K1 as m:
        k1_setup(m, "D4")
        for rid in ko:
            m.reactions.get_by_id(rid).bounds = (0., 0.)
        forced(m, BIO, IGG, CAP, f"{label}, growth capped at {CAP}")

   wild type, growth capped at 0.0125     mu 0.012500000   qmAb    0.0000   interval width 5.6e-05


   BCAT1 knockout, growth capped at 0.0125 mu 0.012500000   qmAb   42.7884   interval width 1.9e-05


## Step 3 - CHO-K1 with BCAT2 added

BCAT2 is absent from this reconstruction, consistent with *Bcat2* not being
expressed in CHO. Adding it back gives the surplus leucine a route that bypasses
IgG, and the coupling is lost once its capacity reaches that surplus.


In [4]:
MU52 = 0.012369243
for frac in (0.0, 0.05, 0.10, 0.25, 0.50, 1.00):
    with K1 as m:
        k1_setup(m, "D4")
        scenario52(m, "D4")
        add_bcat2(m, frac * UPTAKE["D4"])
        forced(m, BIO, IGG, MU52, f"BCAT2 capacity {frac*100:5.1f}% of uptake")

surplus = UPTAKE["D4"] - 0.540177 * MU52
print(f"\n   leucine surplus at this growth rate: {surplus:.6f}"
      f"  = {100*surplus/UPTAKE['D4']:.1f}% of uptake")
print("   BCAT2 abolishes the coupling once it can carry that surplus.")
print("   Measured Bcat2 in IgG-expressing CHO-K1: 0.09 RPKM against Bcat1 31.26,")
print("   i.e. 0.29% of BCAT1, two orders of magnitude below this threshold.")

   BCAT2 capacity   0.0% of uptake        mu 0.012369243   qmAb   43.6783   interval width 7.5e-13


   BCAT2 capacity   5.0% of uptake        mu 0.012369243   qmAb   37.2850   interval width 6.4e-06


   BCAT2 capacity  10.0% of uptake        mu 0.012369243   qmAb   30.8917   interval width 1.3e-05


   BCAT2 capacity  25.0% of uptake        mu 0.012369243   qmAb   11.7118   interval width 3.2e-05


   BCAT2 capacity  50.0% of uptake        mu 0.012369243   qmAb    0.0000   interval width 4.6e-05


   BCAT2 capacity 100.0% of uptake        mu 0.012369243   qmAb    0.0000   interval width 5.4e-05

   leucine surplus at this growth rate: 0.003467  = 34.2% of uptake
   BCAT2 abolishes the coupling once it can carry that surplus.
   Measured Bcat2 in IgG-expressing CHO-K1: 0.09 RPKM against Bcat1 31.26,
   i.e. 0.29% of BCAT1, two orders of magnitude below this threshold.


## Step 4 - iCHO3K, scenario 52

The same seven regulations in a more recent reconstruction. The design does not
transfer: growth is not restricted and no antibody is forced at any growth rate.


In [5]:
I3BIO, I3IGG = "biomass_cho_prod", "DM_igg_g"

def i3_setup(m, ds):
    """2022 boundary bounds mapped onto iCHO3K by exchange identifier.

    358 of 389 map; the 31 that do not are listed in the deposit README.
    """
    have = {r.id for r in m.reactions}
    for j, rid in enumerate(REF):
        if rid.startswith("EX_") and rid.rstrip("_") in have:
            m.reactions.get_by_id(rid.rstrip("_")).bounds = (float(BND[ds][j, 0]),
                                                             float(BND[ds][j, 1]))
    for rid in ("biomass_cho", "biomass_cho_s"):
        m.reactions.get_by_id(rid).bounds = (0., 0.)
    m.reactions.get_by_id(I3IGG).bounds = (0., 1000.)
    m.objective = m.reactions.get_by_id(I3BIO)

with I3 as m:
    i3_setup(m, "D4")
    scenario52(m, "D4")
    mumax = m.slim_optimize()
    print(f"   maximum growth under scenario 52: {mumax:.9f}")
    for f in (0.0, 0.5, 1.0):
        with m as mm:
            forced(mm, I3BIO, I3IGG, f * mumax, f"growth at {f*100:5.1f}% of maximum")

   note: r1109 is irreversible here, so the r1109_b knockout is inert


   maximum growth under scenario 52: 0.044050233


   growth at   0.0% of maximum            mu 0.000000000   qmAb    0.0000   interval width 1.6e-04


   growth at  50.0% of maximum            mu 0.022025116   qmAb    0.0000   interval width 1.1e-04


   growth at 100.0% of maximum            mu 0.044050233   qmAb    0.0000   interval width 8.1e-13


## Step 5 - iCHO3K, scenario 52 with the additional knockouts

iCHO3K carries an active mitochondrial BCAT2 (`LEUTAm`) and three leucine sinks
left at default bounds: the Phe-Leu dipeptide export, apolipoprotein A1 assembly
and glycogenin assembly. Removing those four restores the coupling.


In [6]:
SINKS = ("EX_pheleu_e", "HMR_5166", "r1402")   # dipeptide, apoA1, glycogenin

with I3 as m:
    i3_setup(m, "D4")
    scenario52(m, "D4")
    for rid in ("LEUTAm",) + SINKS:            # BCAT2 and the three leucine sinks
        m.reactions.get_by_id(rid).bounds = (0., 0.)
    mumax = m.slim_optimize()
    print(f"   maximum growth: {mumax:.9f}")
    for f in (0.0, 0.25, 0.5, 1.0):
        with m as mm:
            forced(mm, I3BIO, I3IGG, f * mumax, f"growth at {f*100:5.1f}% of maximum")

   note: r1109 is irreversible here, so the r1109_b knockout is inert
   maximum growth: 0.038931714


   growth at   0.0% of maximum            mu 0.000000000   qmAb  127.8662   interval width 4.8e-05


   growth at  25.0% of maximum            mu 0.009732928   qmAb   73.6875   interval width 7.2e-05
   growth at  50.0% of maximum            mu 0.019465857   qmAb   19.5088   interval width 9.6e-05


   growth at 100.0% of maximum            mu 0.038931714   qmAb    0.0000   interval width 6.6e-05


## Step 6 - iCHO3K coupling

Fourteen plasma-protein assembly reactions in iCHO3K consume cytosolic leucine, so
flux variability alone would suggest fourteen escape routes. Twelve are futile
cycles: a paired degradation reaction returns exactly the leucine consumed, giving
no net drain. Only apolipoprotein A1 (secreted) and glycogenin (leucine retained in
the glycogen primer) are genuine sinks, alongside the Phe-Leu dipeptide export.
Adding each assembly reaction in turn to the double knockout confirms that only
`r1402` changes the outcome.

Each route is then reopened to a capacity expressed as leucine drain (leucine
coefficient x flux) as a percentage of the dataset's forced minimum uptake. Forced
IgG falls in proportion to the capacity granted and survives to 99%, so the result
is sensitive to reactions left effectively unbounded during curation rather than to
alternative metabolism carrying plausible flux.


In [7]:
# The fourteen plasma-protein assembly reactions that consume cytosolic leucine.
PROTEINS = ["HMR_5166","r1092","r1093","r1094","r1095","r1098","r1099","r1100",
            "r1101","r1103","r1104","r1105","r1112","r1402"]
# Each is paired with a degradation reaction that returns its amino acids.
DEGRADER = {"HMR_5166":"r0808","r1092":"r1333","r1093":"r1335","r1094":"r1337",
            "r1095":"r1341","r1098":"r1343","r1099":"r1344","r1100":"r1345",
            "r1101":"r1348","r1103":"r1352","r1104":"r1354","r1105":"r1356",
            "r1112":"r1346"}

leu_c = I3.metabolites.leu_L_c
print("net leucine per unit flux (assembly minus its degrader):")
for rid, deg in DEGRADER.items():
    a = -I3.reactions.get_by_id(rid).metabolites[leu_c]
    b = I3.reactions.get_by_id(deg).metabolites.get(leu_c, 0.0)
    print(f"   {rid:10s} consumes {a:6.1f}, {deg:7s} returns {b:6.1f}"
          f"  ->  net {a-b:6.1f}  {'DRAIN' if abs(a-b) > 1e-9 else 'futile cycle'}")

# r1402 (glycogenin) has no degrader: leucine enters the glycogen primer and stays.
print("\nadding one assembly reaction at a time to BCAT1+BCAT2 KO + the two exports:")
base_ko = ("LEUTA", "LEUTAm", "EX_pheleu_e", "EX_HC00004_e")
with I3 as m:
    i3_setup(m, "D4")
    for rid in base_ko: m.reactions.get_by_id(rid).bounds = (0., 0.)
    print(f"   {'(none)':12s} min IgG at mu = 0: {forced(m, I3BIO, I3IGG, 0.0, 'two exports only'):.4e}")
for rid in PROTEINS:
    if rid == "HMR_5166": continue
    with I3 as m:
        i3_setup(m, "D4")
        for r2 in base_ko + (rid,): m.reactions.get_by_id(r2).bounds = (0., 0.)
        m.reactions.get_by_id(I3BIO).bounds = (0., 0.)
        m.objective = m.reactions.get_by_id(I3IGG); m.objective.direction = "min"
        v = m.slim_optimize()
        flag = "   <-- restores coupling" if (v or 0) > 1e-12 else ""
        print(f"   + {rid:10s} min IgG at mu = 0: {v:.4e}{flag}")

SINKS = {"EX_pheleu_e": 1.0, "HMR_5166": 41.0, "r1402": 39.0}   # leucine per unit flux

def i3_closed(m, ds):
    """BCAT1, BCAT2 and all three leucine sinks closed."""
    i3_setup(m, ds)
    for rid in ("LEUTA", "LEUTAm") + tuple(SINKS): m.reactions.get_by_id(rid).bounds = (0., 0.)

for ds in ("D4", "D2"):
    lm = UPTAKE[ds]
    print(f"--- {ds}: forced minimum leucine uptake {lm:.6f}")
    for rid, per in SINKS.items():
        row = []
        for frac in (0.50, 0.95, 0.99, 1.00):
            with I3 as m:
                i3_closed(m, ds)
                m.reactions.get_by_id(rid).bounds = (0., frac * lm / per)
                m.reactions.get_by_id(I3BIO).bounds = (0., 0.)
                m.objective = m.reactions.get_by_id(I3IGG); m.objective.direction = "min"
                v = m.slim_optimize() or 0.0
            row.append(f"{frac*100:5.0f}%: {v*QCONV:8.3f}")
        print(f"   {rid:12s} forced qmAb at mu = 0, by capacity granted   " + "  ".join(row))

    # growth rate below which IgG is forced, all three closed
    with I3 as m:
        i3_closed(m, ds)
        mumax = m.slim_optimize()
        lo, hi = 0.0, mumax
        for _ in range(30):
            mid = (lo + hi) / 2
            with m as mm:
                mm.reactions.get_by_id(I3BIO).bounds = (mid, mid)
                mm.objective = mm.reactions.get_by_id(I3IGG); mm.objective.direction = "min"
                v = mm.slim_optimize()
            if (v or 0) > 1e-12: lo = mid
            else: hi = mid
    print(f"   IgG forced below mu = {lo:.6f}  (max growth {mumax:.6f})\n")


net leucine per unit flux (assembly minus its degrader):
   HMR_5166   consumes   41.0, r0808   returns    0.0  ->  net   41.0  DRAIN
   r1092      consumes   64.0, r1333   returns   64.0  ->  net    0.0  futile cycle
   r1093      consumes   59.0, r1335   returns   59.0  ->  net    0.0  futile cycle
   r1094      consumes   51.0, r1337   returns   51.0  ->  net    0.0  futile cycle
   r1095      consumes  533.0, r1341   returns  533.0  ->  net    0.0  futile cycle
   r1098      consumes   12.0, r1343   returns   12.0  ->  net    0.0  futile cycle
   r1099      consumes   15.0, r1344   returns   15.0  ->  net    0.0  futile cycle
   r1100      consumes   11.0, r1345   returns   11.0  ->  net    0.0  futile cycle
   r1101      consumes  166.0, r1348   returns  166.0  ->  net    0.0  futile cycle
   r1103      consumes   48.0, r1352   returns   48.0  ->  net    0.0  futile cycle
   r1104      consumes   51.0, r1354   returns   51.0  ->  net    0.0  futile cycle
   r1105      consumes   6

   two exports only                       mu 0.000000000   qmAb    0.0000   interval width 1.6e-04


   (none)       min IgG at mu = 0: 0.0000e+00
   + r1092      min IgG at mu = 0: 0.0000e+00
   + r1093      min IgG at mu = 0: 0.0000e+00
   + r1094      min IgG at mu = 0: 0.0000e+00
   + r1095      min IgG at mu = 0: 0.0000e+00
   + r1098      min IgG at mu = 0: 0.0000e+00


   + r1099      min IgG at mu = 0: 0.0000e+00
   + r1100      min IgG at mu = 0: 0.0000e+00


   + r1101      min IgG at mu = 0: 0.0000e+00
   + r1103      min IgG at mu = 0: 0.0000e+00


   + r1104      min IgG at mu = 0: 0.0000e+00
   + r1105      min IgG at mu = 0: 0.0000e+00


... (5 outputs left out)

   EX_pheleu_e  forced qmAb at mu = 0, by capacity granted      50%:   54.134     95%:    5.413     99%:    1.083    100%:    0.000


   HMR_5166     forced qmAb at mu = 0, by capacity granted      50%:   54.134     95%:    5.413     99%:    1.083    100%:    0.000


   r1402        forced qmAb at mu = 0, by capacity granted      50%:   54.134     95%:    5.413     99%:    1.083    100%:    0.000


   IgG forced below mu = 0.019450  (max growth 0.038932)



## Step 7 - Leucine thresholds for coupling

With BCAT1 knocked out, IgG becomes obligatory below the growth rate at which biomass
can no longer absorb the forced leucine uptake. The bisected threshold is compared
with the closed form, uptake divided by the leucine coefficient of biomass, and then
against a varied uptake bound, since that bound is taken from a published dataset
rather than measured here. The final block reports how many CHO-K1 exchange bounds
have a counterpart in iCHO3K.


In [8]:
leu_coeff = abs(K1.reactions.get_by_id(BIO).metabolites[K1.metabolites.get_by_id("leu_L[c]")])
print(f"leucine coefficient of {BIO}: {leu_coeff:.6f}\n")

for ds in ("D2", "D4"):
    with K1 as m:
        k1_setup(m, ds)                      # blocks biomass_cho
        m.reactions.get_by_id("LEUTA").bounds = (0., 0.)
        mumax = m.slim_optimize()
        lo, hi = 0.0, mumax
        for _ in range(40):
            mid = (lo + hi) / 2
            with m as mm:
                mm.reactions.get_by_id(BIO).bounds = (mid, mid)
                mm.objective = mm.reactions.get_by_id(IGG); mm.objective.direction = "min"
                v = mm.slim_optimize()
            if (v or 0) > 1e-12: lo = mid
            else: hi = mid
    print(f"{ds}: forced leucine uptake {UPTAKE[ds]:.6f} | IgG forced below mu = {lo:.6f}"
          f" | uptake/coefficient = {UPTAKE[ds]/leu_coeff:.6f}")

# How many of the CHO-K1 exchange bounds have a counterpart in iCHO3K.
have = {r.id for r in I3.reactions}
bnds = {rid.rstrip("_") for rid in REF if rid.startswith("EX_")}
unmapped = sorted(k for k in bnds if k not in have)
print(f"\nCHO-K1 exchanges {len(bnds)}; mapped onto iCHO3K {len(bnds)-len(unmapped)}; "
      f"unmapped {len(unmapped)}:")
print("   " + ", ".join(unmapped))

# The bound is taken from a published dataset, so how far does the threshold
# depend on it? Same bisection, with the imposed minimum uptake scaled.
LEU = REF.index("EX_leu_L_e_")
print("\nthreshold against the imposed minimum uptake")
for ds in ("D4", "D2"):
    lb0, ub0 = float(BND[ds][LEU, 0]), float(BND[ds][LEU, 1])
    for frac in (1.00, 0.50):
        with K1 as m:
            k1_setup(m, ds)
            m.reactions.get_by_id("EX_leu_L_e_").bounds = (lb0, ub0 * frac)
            m.reactions.get_by_id("LEUTA").bounds = (0., 0.)
            lo, hi = 0.0, m.slim_optimize()
            for _ in range(30):
                mid = (lo + hi) / 2
                with m as mm:
                    mm.reactions.get_by_id(BIO).bounds = (mid, mid)
                    mm.objective = mm.reactions.get_by_id(IGG)
                    mm.objective.direction = "min"
                    v = mm.slim_optimize()
                if (v or 0) > 1e-12: lo = mid
                else: hi = mid
        print(f"   {ds} at {frac:.0%} of the bound: forced below mu = {lo:.6f}"
              f"   closed form {UPTAKE[ds]*frac/leu_coeff:.6f}")

# Specific leucine uptake measured in the fed-batch cultures, phase means over
# days 5-14, converted to the threshold each would imply.
MEASURED = {"CHO": 0.005600, "CHO'": 0.006122, "CHO-B": 0.005900}
print("\nthreshold implied by the measured specific uptake")
for name, q in MEASURED.items():
    print(f"   {name:6s} {q:.6f} mmol/gDCW/h  ->  IgG forced below mu = {q/leu_coeff:.6f}")


leucine coefficient of biomass_cho_producing: 0.540177



D2: forced leucine uptake 0.008593 | IgG forced below mu = 0.015907 | uptake/coefficient = 0.015907


D4: forced leucine uptake 0.010148 | IgG forced below mu = 0.018787 | uptake/coefficient = 0.018787

CHO-K1 exchanges 389; mapped onto iCHO3K 358; unmapped 31:
   EX_C05965_e, EX_CLPND_e, EX_HC00229_e, EX_HC00955_e, EX_HC01577_e, EX_HC01610_e, EX_HC02176_e, EX_HC02203_e, EX_HC02204_e, EX_HC02205_e, EX_HC02207_e, EX_ak2lgchol_cho_e, EX_avite1_e, EX_avite2_e, EX_crm_cho_e, EX_crmp_cho_e, EX_dag_cho_e, EX_galside_cho_e, EX_ivcrn, EX_lpchol_cho_e, EX_mag_cho_e, EX_paf_cho_e, EX_pchol_cho_e, EX_pe_cho_e, EX_pglyc_cho_e, EX_ps_cho_e, EX_spc_cho_e, EX_tag_cho_e, EX_xolest2_cho_e, EX_xolest_cho_e, EX_yvite_e

threshold against the imposed minimum uptake


   D4 at 100% of the bound: forced below mu = 0.018787   closed form 0.018787


   D4 at 50% of the bound: forced below mu = 0.009393   closed form 0.009393


   D2 at 100% of the bound: forced below mu = 0.015907   closed form 0.015907


   D2 at 50% of the bound: forced below mu = 0.007954   closed form 0.007954

threshold implied by the measured specific uptake
   CHO    0.005600 mmol/gDCW/h  ->  IgG forced below mu = 0.010367
   CHO'   0.006122 mmol/gDCW/h  ->  IgG forced below mu = 0.011333
   CHO-B  0.005900 mmol/gDCW/h  ->  IgG forced below mu = 0.010922


## Step 8 - Sensitivity to the upregulation coefficient

The growth suppression in scenario 52 comes from the forced backward flux in
RTOTAL2t and RTOTAL3t, whose magnitude is set by the upregulation coefficient. This
establishes the range of that coefficient over which the design is feasible at all,
and how the coupling behaves across it, together with the same design evaluated
under each of the four dataset bound sets.


In [9]:
def s52_eup(m, ds, e_up):
    for rid in ("THRA", "AACOAT", "LEUTA", "PSP_L"):
        m.reactions.get_by_id(rid).bounds = (0., 0.)
    r = m.reactions.get_by_id("r1109")
    if r.lower_bound < 0:
        r.lower_bound = 0.
    for rid in ("RTOTAL2t", "RTOTAL3t"):
        i = REF.index(rid)
        lb, ub = float(BND[ds][i, 0]), float(BND[ds][i, 1])
        m.reactions.get_by_id(rid).bounds = (lb, ub + e_up * lb)


print("e_up sweep: growth optimum and whether IgG is forced there\n")
for ds in ("D4", "D2"):
    print(f"  {ds}")
    for e in (0.40, 0.50, 0.55, 0.60, 0.605, 0.61, 0.62, 0.65, 1.10):
        with K1 as m:
            k1_setup(m, ds); s52_eup(m, ds, e)
            mu = m.slim_optimize()
            if mu is None or mu != mu:
                print(f"    e_up {e:5.3f}  infeasible"); continue
            try:
                lo = forced(m, BIO, IGG, mu, f"    e_up {e:5.3f}")
            except Exception:
                print(f"    e_up {e:5.3f}  mu {mu:.6f}  infeasible at its optimum")

print("\ndataset bounds at e_up = 0.60")
for ds in ("D1", "D2", "D3", "D4"):
    with K1 as m:
        k1_setup(m, ds); s52_eup(m, ds, 0.60)
        mu = m.slim_optimize()
        if mu is None or mu != mu:
            print(f"  {ds}: infeasible"); continue
        try:
            forced(m, BIO, IGG, mu, f"  {ds}")
        except Exception:
            print(f"  {ds}: mu {mu:.6f}, infeasible at its optimum")
        # Threshold: the largest growth rate at which IgG is still forced, bisected
        # as in Step 7. Where the regulation set's own optimum lies below the
        # leucine threshold, the optimum is what binds and the two coincide.
        lo, hi = 0.0, mu
        for _ in range(40):
            mid = (lo + hi) / 2
            with m as mm:
                mm.reactions.get_by_id(BIO).bounds = (mid, mid)
                mm.objective = mm.reactions.get_by_id(IGG); mm.objective.direction = "min"
                v = mm.slim_optimize()
            if (v or 0) > 1e-12: lo = mid
            else: hi = mid
        print(f"       threshold mu {lo:.6f}")


e_up sweep: growth optimum and whether IgG is forced there

  D4


       e_up 0.400                         mu 0.029704945   qmAb    0.0000   interval width 9.8e-14


       e_up 0.500                         mu 0.029704945   qmAb    0.0000   interval width 9.8e-14


       e_up 0.550                         mu 0.029704945   qmAb    0.0000   interval width 9.8e-14


       e_up 0.600                         mu 0.012369243   qmAb   43.6783   interval width 9.8e-14


       e_up 0.605                         mu 0.006040140   qmAb   86.7556   interval width 4.8e-14
    e_up 0.610  infeasible
    e_up 0.620  infeasible


... (11 outputs left out)

     D3                                   mu 0.019201045   qmAb    0.0000   interval width 1.6e-13


       threshold mu 0.011162


     D4                                   mu 0.012369243   qmAb   43.6783   interval width 9.8e-14


       threshold mu 0.012369


## Step 9 - Growth coupling across all scenarios

Every regulation set is applied in turn and tested by the criterion used for
scenario 52: the minimum IgG flux at the scenario's own growth optimum, a minimum
above zero meaning production is obligatory rather than merely permitted. All 89
regulation sets are tested, which takes about ten minutes; one parameter in the cell
reduces this to every fourth scenario for a quicker check.


In [10]:
import openpyxl

IDX = {r: i for i, r in enumerate(REF)}


def apply_regulations(m, ds, reg, e_up):
    """GAMS splits reversible reactions into v = v_f - v_b; 189 are reversed and
    carry _r. Regulations act on the splits, so they are collected in split space
    and projected back to a net bound."""
    groups = {}
    for tok in [t.strip() for t in str(reg).split(";") if t.strip()]:
        name, _, act = tok.partition(":")
        name, act, base, side = name.strip(), act.strip().lower(), name.strip(), ""
        for s in ("_f", "_b", "_r"):
            if name.endswith(s) and name[:-2] in IDX:
                base, side = name[:-2], s; break
        if base in IDX:
            groups.setdefault(base, []).append((side, act))
    for base, acts in groups.items():
        i = IDX[base]
        lb0, ub0 = float(BND[ds][i, 0]), float(BND[ds][i, 1])
        f_lo, f_hi, b_lo, b_hi = 0., max(ub0, 0.), 0., max(-lb0, 0.)
        for side, act in acts:
            if act == "ko":
                if side == "_f":   f_hi = 0.
                elif side == "_b": b_hi = 0.
                else:              f_hi = b_hi = 0.
            elif act == "up":
                if side == "_b" or (side == "_r" and ub0 <= 0):
                    b_lo = e_up * max(-lb0, 0.)
                else:
                    f_lo = e_up * max(ub0, 0.)
        f_lo, b_lo = min(f_lo, f_hi), min(b_lo, b_hi)
        lo, hi = f_lo - b_hi, f_hi - b_lo
        m.reactions.get_by_id(base).bounds = (0., 0.) if lo > hi else (lo, hi)


ws = openpyxl.load_workbook(DEP / "scenarios.xlsx", data_only=True)["93_initial"]
h = [c.value for c in ws[1]]
col = {k: h.index(k) for k in ("scenario_id", "dataset", "algorithm", "e_up",
                               "regulations", "mu_predicted")}
# All 89 scenarios, which takes roughly ten minutes and is the run reported in the
# paper. Set EVERY = 4 to test every fourth instead, a three-minute spot check.
EVERY = 1

tally, repro = {"Bio_Igg": [0, 0], "Igg_Igg": [0, 0]}, [0, 0]
seen = 0
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[col["scenario_id"]] is None:
        continue
    seen += 1
    if (seen - 1) % EVERY:
        continue
    ds, alg = row[col["dataset"]], row[col["algorithm"]]
    with K1 as m:
        k1_setup(m, ds)
        apply_regulations(m, ds, row[col["regulations"]], float(row[col["e_up"]]))
        mu = m.slim_optimize()
        if mu is None or mu != mu:
            continue
        tally[alg][1] += 1
        try:
            if min_igg(m, BIO, IGG, mu) > 1e-9:
                tally[alg][0] += 1
        except Exception:
            pass
    pub = row[col["mu_predicted"]]
    if pub is not None:
        repro[1] += 1
        repro[0] += abs(mu - float(pub)) / max(float(pub), 1e-12) < 0.02

print(f"  tested {sum(v[1] for v in tally.values())} of {seen} scenarios"
      f"{', every ' + str(EVERY) + 'th' if EVERY > 1 else ''}")
for alg, (c, n) in tally.items():
    print(f"  {alg}: {c} of {n} force IgG at their own growth optimum")
print(f"  mu reproduces the published value for {repro[0]} of {repro[1]} scenarios")


  tested 89 of 89 scenarios
  Bio_Igg: 51 of 51 force IgG at their own growth optimum
  Igg_Igg: 38 of 38 force IgG at their own growth optimum
  mu reproduces the published value for 85 of 89 scenarios
